# Unified Adaptable Masking

- TODO: expand bullets
- objective: [graph](./masking.pdf)

In [1]:
import pathlib as pth
import tensorflow as tf
from datetime import datetime
ks = tf.keras
kl = ks.layers

- load our meta-data

In [2]:
vocab = ('x', 'y', '+', '-', '*', '=', ',', ':')
vocab += ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

tokens = {k: v for v, k in enumerate(vocab, start=5)}
tokens.update({v: k for k, v in tokens.items()})

SEP = tokens[':']

- get paths to the file shards
- recast our parsed streams and start using `RaggedTensors` instead of sparse ones
- before handing the streams of data to Keras convert them (for now) to dense tensors

In [3]:
def paths(ps):
    d = pth.Path('/tmp/q/dataset')
    for i in range(ps.num_shards):
        i = '{:0>4d}'.format(i)
        yield str(d / f'shard_{i}.tfrecords')

@tf.function
def caster(d):
    return {k: tf.cast(v, tf.int32) for k, v in d.items()}

@tf.function
def adapter(d, len_max_input):
    ds = tf.RaggedTensor.from_sparse(d['defs'])
    ss = tf.fill([ds.nrows(), 1], SEP)
    os = tf.RaggedTensor.from_sparse(d['op'])
    x = tf.concat([ds, ss, os], axis=1).to_tensor()
    x = tf.pad(x, [[0, 0], [0, len_max_input - tf.shape(x)[-1]]])
    y = tf.RaggedTensor.from_sparse(d['res'])[:, :1].to_tensor()
    return x, y

- ready to create our dataset

In [4]:
def dset_for(ps):
    ds = tf.data.TFRecordDataset(list(paths(ps))).batch(ps.dim_batch)
    fs = {
        'defs': tf.io.VarLenFeature(tf.int64),
        'op': tf.io.VarLenFeature(tf.int64),
        'res': tf.io.VarLenFeature(tf.int64),
    }
    ds = ds.map(lambda x: tf.io.parse_example(x, fs)).map(caster)
    return ds.map(lambda d: adapter(d, tf.constant(ps.len_max_input)))

- we need to tell our Keras layers to support masking, let's do it once for all of them
- our first layer, the one to calculate the masking tensor, has to override `compute_mask`
- we could also transfer the mask calculation to a layer that would do it as a side-effect
- in that case we would use the 2 commented out lines 

In [5]:
class Layer(kl.Layer):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.supports_masking = True

class Masking(Layer):
    def __init__(self):
        super().__init__()
        # self._compute_output_and_mask_jointly = True

    def compute_mask(self, x, mask=None):
        return tf.not_equal(x, 0)

    def call(self, x):
        # x._keras_mask = self.compute_mask(x)
        return x

- our embedding layer is as simple as it gets: it creates the embedding table and then does the actual lookup
- once the embedded values are determined, we then apply masking cleanly
- Keras knows that we want to use the mask tensor from us listing the `mask=None` keyword argument
- for `autograph`'s sake we need to explicitly check that the optional `mask` argument is not `None`

In [6]:
class Embed(Layer):
    def __init__(self, ps):
        super().__init__(dtype=tf.float32)
        s = (ps.dim_vocab, ps.dim_hidden)
        self.emb_t = self.add_weight(name='emb_t', shape=s)

    def call(self, x, mask=None):
        y = tf.nn.embedding_lookup(self.emb_t, x)
        if mask is not None:
            y *= tf.cast(mask, tf.float32)[:, :, None]
        return y

- our self-attention layer, called `Reflect`, does the absolute minimum required steps to implement the "attention" mechanism of the `transformer` architecture
- an excellent, creative explanation of how it works is at http://jalammar.github.io/illustrated-transformer/
- please note the masking tensor being automatically supplied to the call by Keras
- we only need to state our intention to mask by adding the `mask=None` keyword argument
- the actual masking calculation, based on our previously created boolean tensor, is now trivial

In [7]:
class Reflect(Layer):
    def build(self, shape):
        s = shape[-1]
        self.scale = 1 / (s**0.5)
        self.q_w = self.add_weight(name='q_w', shape=(s, s))
        self.k_w = self.add_weight(name='k_w', shape=(s, s))
        self.v_w = self.add_weight(name='v_w', shape=(s, s))
        return super().build(shape)

    def call(self, x, mask=None):
        q = tf.einsum('bsi,ij->bsj', x, self.q_w)
        k = tf.einsum('bsi,ij->bsj', x, self.k_w)
        y = tf.einsum('bsi,bzi->bsz', q, k) * self.scale
        if mask is not None:
            # tf.print(' *** applying mask')
            m = tf.logical_not(mask)
            m = tf.cast(m, tf.float32)[:, :, None]
            y += m * -1e9
        v = tf.einsum('bsi,ij->bsj', x, self.v_w)
        y = tf.einsum('bsz,bzi->bsi', tf.nn.softmax(y), v)
        return y

- now we are ready to create and compile our Keras `functional` model
- as the objective of this blog is to showcase masking, all the other necessary "plumbing" layers are the canned Keras variety ones


In [8]:
def model_for(ps):
    x = ks.Input(shape=(ps.len_max_input, ), dtype='int32')
    y = Masking()(x)
    y = Embed(ps)(y)
    y = Reflect()(y)
    y = kl.Reshape((ps.len_max_input * ps.dim_hidden, ))(y)
    y = kl.Dense(ps.dim_dense, activation='relu')(y)
    y = kl.Dense(ps.dim_vocab, name='out', activation=None)(y)
    m = ks.Model(inputs=x, outputs=y)
    m.compile(optimizer=ps.optimizer, loss=ps.loss, metrics=[ps.metrics])
    print(m.summary())
    return m

- our parameters have slightly increased in number
- please see the previous blogs for the justification of the scheme

In [9]:
params = dict(
    dim_batch=2,
    dim_dense=150,
    dim_hidden=15,
    dim_vocab=len(vocab) + 5,
    len_max_input=20,
    loss=ks.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=ks.metrics.SparseCategoricalAccuracy(),
    num_shards=2,
    optimizer=ks.optimizers.Adam(),
)

class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

- once we instantiate our params and our dataset, and using the already compiled model, we are ready to start a training session
- our aim is to use as much of the great functionality and error checking that Keras provides, so using the model's `fit` method is all we need for now

In [13]:
ps = Params(**params)
ds = dset_for(ps)
m = model_for(ps)
ld = datetime.now().strftime('%Y%m%d-%H%M%S')
ld = f'/tmp/q/logs/{ld}'
cs = [ks.callbacks.TensorBoard(log_dir=ld, histogram_freq=1)]
m.fit(ds, callbacks=cs, epochs=10)

W0624 21:33:05.619439 140250756195968 training_utils.py:1444] Expected a shuffled dataset but input dataset `x` is not shuffled. Please invoke `shuffle()` on input dataset.
W0624 21:33:05.799244 140250756195968 summary_ops_v2.py:1110] Model failed to serialize as JSON. Ignoring... Layers with arguments in `__init__` must override `get_config`.


Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 20)]              0         
_________________________________________________________________
masking_1 (Masking)          (None, 20)                0         
_________________________________________________________________
embed_1 (Embed)              (None, 20, 15)            345       
_________________________________________________________________
reflect_1 (Reflect)          (None, 20, 15)            675       
_________________________________________________________________
reshape_1 (Reshape)          (None, 300)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 150)               45150     
_________________________________________________________________
out (Dense)                  (None, 23)                3473

In [11]:
%load_ext tensorboard

- with our TensorBoard `callback` in place, the model's `fit` method will generate the standard summaries
- if you haven't run the code, an already generated graph is [here](./masking.pdf)

In [12]:
%tensorboard --logdir /tmp/q/logs

Reusing TensorBoard on port 6007 (pid 20958), started 10:10:53 ago. (Use '!kill 20958' to kill it.)